A code-review agent needs to run several tasks. Some can start immediately. Others depend on earlier results. For example, the agent can fetch a code diff and load the review policy independently, but it needs both before it can review the code.

Implement this function:

def execution_order(dependencies: dict[str, list[str]]) -> list[str]:
    ...

Each key is a task. Its value lists the tasks that must finish before it can start. Return an order that runs every task exactly once. If several tasks are ready, choose the alphabetically smallest currently ready task.

dependencies = {
    "fetch_diff": [],
    "load_policy": [],
    "review_code": ["fetch_diff", "load_policy"],
    "check_secrets": ["fetch_diff"],
}

Expected Output: 
["fetch_diff", "check_secrets", "load_policy", "review_code"]

Initially, fetch_diff and load_policy are ready, so fetch_diff goes first. Completing it makes check_secrets ready. Now the choice is between check_secrets and load_policy, so check_secrets goes next.

Contract
- Empty input returns [].
- A prerequisite missing from the dictionary raises ValueError.
- A cycle, including a task depending on itself, raises ValueError.
- Every prerequisite list contains unique names. Assume up to 100,000 tasks and 300,000 dependency entries. Preserve the input.


Use Kahn’s topological sort with a min-heap. Your idea to take a ready task and append it to task_order is the core of the algorithm. The queue choice depends on what the interviewer asks for:

| Requirement | How to select the next ready task | Time |
|---|---|---|
| Return **any** valid order | `deque.popleft()` | `O(V + E)` |
| Return the **alphabetically smallest currently ready** task at every step | `heapq.heappop(ready)` | `O(E + V log V)` |

Here V is the number of tasks and E is the total number of prerequisite entries. The second row is our exercise.

How to derive it
The input tells you what each task is waiting for. For example, dependencies["review_code"] = ["fetch_diff", "load_policy"].

When fetch_diff finishes, you need to answer the question in the opposite direction: who was waiting for fetch_diff? You could scan the input after every completion. Building that reverse lookup once saves the repeated scans.

Keep two dictionaries:
- remaining[task] tells you how many prerequisites are still unfinished. A task becomes ready exactly when this count reaches zero.
- dependents[task] tells you whose counts to reduce when this task finishes. For example, dependents["fetch_diff"] contains "review_code" and "check_secrets".



In [11]:
from heapq import heapify, heappop, heappush


def execution_order(dependencies: dict[str, list[str]]) -> list[str]:
    remaining = {task: len(prereqs) for task, prereqs in dependencies.items()}
    dependents = {task: [] for task in dependencies}

    for task, prereqs in dependencies.items():
        for prereq in prereqs:
            if prereq not in dependencies:
                raise ValueError(f"Unknown prerequisite {prereq!r} for {task!r}")
            dependents[prereq].append(task)

    ready = [task for task, count in remaining.items() if count == 0]
    heapify(ready)
    task_order = []

    while ready:
        task = heappop(ready)
        task_order.append(task)

        for dependent in dependents[task]:
            remaining[dependent] -= 1
            if remaining[dependent] == 0:
                heappush(ready, dependent)

    if len(task_order) != len(dependencies):
        raise ValueError("Workflow contains a dependency cycle")

    return task_order

# Usage

dependencies = {
    "fetch_diff": [],
    "load_policy": [],
    "review_code": ["fetch_diff", "load_policy"],
    "check_secrets": ["fetch_diff"],
}



In [12]:
import pytest


def test_newly_ready_task_takes_alphabetical_precedence():
    assert execution_order({"a": [], "z": [], "b": ["a"]}) == ["a", "b", "z"]


def test_empty_workflow():
    assert execution_order({}) == []


@pytest.mark.parametrize("workflow", [
    {"a": ["a"]},                       # Self-cycle
    {"a": ["b"], "b": ["a"], "c": []},  # Cycle alongside a runnable task
])
def test_cycle(workflow):
    with pytest.raises(ValueError, match="cycle"):
        execution_order(workflow)


def test_unknown_prerequisite():
    with pytest.raises(ValueError, match="Unknown prerequisite"):
        execution_order({"a": ["missing"]})

In [ ]:
from collections import deque
def execution_order(dependencies: dict[str, list[str]]): # -> list[str]:
    queue = deque()
    ready = []

    for task, prequisites in dependencies.items():
        # Step 1: Build the ready list, i.e. task with no dependencies will come first
        if not prequisites:
            ready.append(task)

    print (ready)

# usage 
dependencies = {
    "fetch_diff": [],
    "load_policy": [],
    "review_code": ["fetch_diff", "load_policy"],
    "check_secrets": ["fetch_diff"],
}

# execution_order(dependencies)
print (dependencies["review_code"])

prerequisites = dependencies["review_code"]
is_ready = all(task in finished for task in prerequisites)

['fetch_diff', 'load_policy']
